Import des données 
Triangulation de Delauney
Création de la carte Folium
Ajout des stations
Ajout des connexions
Ajout légende

In [ ]:
import folium
import json
import numpy as np
from scipy.spatial import Delaunay
from folium.plugins import FloatImage
import os

# Définir le chemin du fichier dans le dossier parent
file_path = os.path.abspath(os.path.join(os.getcwd(), "..", "velib_stations.json"))

# Charger les données depuis le fichier JSON
with open(file_path, "r", encoding="utf-8") as file:
    stations = json.load(file)["data"]["stations"]

# Extraire les coordonnées (latitude, longitude)
points = np.array([[s["lat"], s["lon"]] for s in stations])

# Effectuer la Triangulation de Delaunay
Triangulation = Delaunay(points)

# Centrer la carte sur Paris
paris_coords = (48.8566, 2.3522)
m = folium.Map(location=paris_coords, zoom_start=12)

# Ajouter les stations Vélib' sur la carte
for s in stations:
    folium.CircleMarker(
        location=[s["lat"], s["lon"]],
        popup=[s["name"], s["capacity"]], max_width=100,
        color="blue",
        radius=s["capacity"]//4,
        fill=True,
        fill_color='blue'
    ).add_to(m)

# Ajouter les Triangles de Delaunay sous forme de lignes
for simplex in Triangulation.simplices:
    point = [points[i] for i in simplex]  # Récupérer les sommets du Triangle
    folium.PolyLine(locations=point + [point[0]], color="red", weight=2).add_to(m)  # Fermer le Triangle



# Ajouter une légende personnalisée
legend_html = """
<div style="
    position: fixed;
    bottom: 50px;
    right: 50px;
    background-color: white;
    padding: 10px;
    border-radius: 8px;
    box-shadow: 2px 2px 6px rgba(0,0,0,0.3);
    font-size: 14px;
    z-index: 999;
">
    <b>Légende :</b><br>
    🚲 <span style="color:blue;">Points Bleus</span> - Stations Vélib'<br>
    🔺 <span style="color:red;">Lignes Rouges</span> - Triangulation de Delaunay
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m.save("velib_delaunay_legend.html")

In [8]:
print(points)

[[48.865983    2.275725  ]
 [48.79892241  2.45374515]
 [48.77819275  2.39630202]
 ...
 [48.87442277  2.32846856]
 [48.87040603  2.32324351]
 [48.8512971   2.3624535 ]]


Création de la liste d'adjacence :

Import des données
Liste des noms des station
Liste des coordonnées
Triangulation de Delauney
Création de la liste
Ajout des connexion dans la liste 
Export en JSON

In [2]:
import json
import numpy as np
from scipy.spatial import Delaunay
import os

# Définir le chemin du fichier dans le dossier parent
file_path = os.path.abspath(os.path.join(os.getcwd(), "..", "velib_stations.json"))

# Charger les données depuis le fichier JSON
with open(file_path, "r", encoding="utf-8") as file:
    stations = json.load(file)["data"]["stations"]

# Extraire les noms des stations et leurs coordonnées (lat, lon)
nom_station = [s["name"] for s in stations]  # Liste des noms
coo = np.array([[s["lat"], s["lon"]] for s in stations])  # Coordonnées

# Effectuer la Tri de Delaunay
Triangulation = Delaunay(coo)

# Initialiser la liste d'adjacence
Liste_adjacence = {name: set() for name in nom_station}  # Dictionnaire de sets

# Remplir la liste d'adjacence à partir des Triangles
for simplex in Triangulation.simplices:  # simplex = un Triangle (3 sommets)
    for i in range(3):  # Parcourir les 3 sommets du Triangle
        for j in range(3):
            if i != j:  # Ne pas ajouter une station comme voisine d'elle-même
                station_i = nom_station[simplex[i]]
                station_j = nom_station[simplex[j]]
                Liste_adjacence[station_i].add(station_j)

# Convertir les sets en listes pour un affichage JSON-friendly
Liste_adjacence = {k: list(v) for k, v in Liste_adjacence.items()}

# Afficher un extrait de la liste d'adjacence
print(json.dumps(Liste_adjacence, indent=4, ensure_ascii=False))

# Sauvegarder la liste d'adjacence dans un fichier JSON
with open("velib_Liste_adjacence.json", "w", encoding="utf-8") as f:
    json.dump(Liste_adjacence, f, indent=4, ensure_ascii=False)


{
    "Benjamin Godard - Victor Hugo": [
        "Mairie du 16ème",
        "Flandrin - Henri Martin",
        "Flandrin - Longchamp",
        "Victor Hugo - La Pompe"
    ],
    "Hôpital Mondor": [
        "Créteil Village",
        "Les Juilliottes",
        "Préfecture de Créteil",
        "Bleuets - Bordières",
        "Centre Hospitalier Intercommunal de Créteil",
        "Iles de Loisirs de Créteil",
        "Liberté - Vert-de-Maisons"
    ],
    "Rouget de L'isle - Watteau": [
        "Conservatoire de Musique",
        "8 Mai 1945 - 10 Juillet 1940",
        "Lebrun - Colonel Fabien ",
        "Camille Risch - Paul Armangot",
        "Balzac - Olympes de Gouges",
        "Youri Gagarine - Commune de Paris"
    ],
    "Toudouze - Clauzel": [
        "Jean-Baptiste Pigalle - La Bruyere",
        "Saint Georges - d'Aumale",
        "Victor Massé - Jean-Baptiste Pigalle",
        "Choron - Martyrs",
        "Lallier - Trudaine "
    ],
    "Cassini - Denfert-Rochereau": [
        "

Optimisation

In [1]:
import folium
import json
import numpy as np
from scipy.spatial import Delaunay
from folium.plugins import FloatImage
import branca.colormap as cm
from math import radians, sin, cos, sqrt, atan2
import os

def repartition(Nv, Cmax, C):
    if Nv == 6:
        return 0
    else:
        return 0.5 * ((Nv - 6)/6) + (1 - 0.5) * ((Cmax - C)/Cmax)
    
def longueur_cable(lat1, lon1, lat2, lon2):
    R = 6371  # Rayon de la Terre en km
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c  # Distance en km

# Définir le chemin du fichier velib
file_path_velib = os.path.abspath(os.path.join(os.getcwd(), "..", "velib_stations.json"))
# Définir le chemin de la liste d'adjacence
file_path_adj = "velib_Liste_adjacence.json"

# Charger les données depuis le fichier JSON
with open(file_path_velib, "r", encoding="utf-8") as file:
    stations = json.load(file)["data"]["stations"]

# Charger la liste d'adjacence
with open(file_path_adj, "r") as f:
    adj_list = json.load(f)

# Extraire les coordonnées (latitude, longitude)
points = np.array([[s["lat"], s["lon"]] for s in stations])

# Effectuer la Triangulation de Delaunay
Triangulation = Delaunay(points)

# Création d'un dictionnaire de correspondance nom -> coordonnées
station_coords = {s["name"]: (s["lat"], s["lon"]) for s in stations}

# Construire la liste des arêtes pondérées
aretes = []
for station, voisins in adj_list.items():
    if station in station_coords:
        lat1, lon1 = station_coords[station]
        for voisin in voisins:
            if voisin in station_coords:
                lat2, lon2 = station_coords[voisin]
                distance = longueur_cable(lat1, lon1, lat2, lon2)
                aretes.append((distance, station, voisin))

# Appliquer l'algorithme de Kruskal
class Kruskal:
    def __init__(self, elements):
        self.parent = {e: e for e in elements}
        self.rank = {e: 0 for e in elements}

    def find(self, item):
        if self.parent[item] != item:
            self.parent[item] = self.find(self.parent[item])  # Compression de chemin
        return self.parent[item]

    def union(self, set1, set2):
        root1 = self.find(set1)
        root2 = self.find(set2)
        if root1 != root2:
            if self.rank[root1] > self.rank[root2]:
                self.parent[root2] = root1
            elif self.rank[root1] < self.rank[root2]:
                self.parent[root1] = root2
            else:
                self.parent[root2] = root1
                self.rank[root1] += 1

# Trier les arêtes par poids croissant
aretes.sort()

# Initialiser Kruskal
stations_names = list(station_coords.keys())
uf = Kruskal(stations_names)
minimum_spanning_tree = []

for weight, u, v in aretes:
    if uf.find(u) != uf.find(v):
        uf.union(u, v)
        minimum_spanning_tree.append((u, v, weight))
        if len(minimum_spanning_tree) == len(stations_names) - 1:
            break

# Centrer la carte sur Paris
paris_coords = (48.8566, 2.3522)
m = folium.Map(location=paris_coords, zoom_start=12)

# Définition d'un colormap (rouge → jaune → vert)
colormap = cm.LinearColormap(
    colors=["red", "yellow", "green"],  # Dégradé
    vmin=-0.5, vmax=1,  # Intervalle de l'indice Ir
    caption="Indice de Performance"
)

Cmax = max(station["capacity"] for station in stations)

# Ajouter le nombre de voisins pour chaque station
for s in stations:
    station_name = s["name"]
    if station_name in adj_list:
        s["Nv"] = len(adj_list[station_name])
    else:
        s["Nv"] = 0
    s["Ir"] = repartition(s["Nv"], Cmax, s["capacity"])

# Ajouter les stations Vélib' sur la carte
for s in stations:
    color = colormap(s["Ir"])  # Associe la couleur en fonction de I_r
    folium.CircleMarker(
        location=[s["lat"], s["lon"]],
        popup=f"Station: {s['name']}<br>Capacité: {s['capacity']}<br>Indice Ir: {s['Ir']:.2f}" ,max_width=250,
        color=color,
        radius=s["capacity"]//4,
        fill=True,
        fill_color=color
    ).add_to(m)

# Ajouter les Triangles de Delaunay sous forme de lignes
for simplex in Triangulation.simplices:
    point = [points[i] for i in simplex]  # Récupérer les sommets du Triangle
    folium.PolyLine(locations=point + [point[0]], color="red", weight=2).add_to(m)  # Fermer le Triangle

# Ajouter les arêtes de l'ACM
for u, v, weight in minimum_spanning_tree:
    lat1, lon1 = station_coords[u]
    lat2, lon2 = station_coords[v]
    folium.PolyLine([(lat1, lon1), (lat2, lon2)], color="green", weight=4).add_to(m)


# Ajouter une légende personnalisée
legend_html = """
<div style="
    position: fixed;
    bottom: 50px;
    right: 50px;
    background-color: white;
    padding: 10px;
    border-radius: 8px;
    box-shadow: 2px 2px 6px rgba(0,0,0,0.3);
    font-size: 14px;
    z-index: 999;
">
    <b>Légende :</b><br>
    🚲 <span style="color:blue;">Points Bleus</span> - Stations Vélib'<br>
    🔺 <span style="color:red;">Lignes Rouges</span> - Triangulation de Delaunay<br>
    🟢 <span style="color:green;">Arbre Couvrant Minimal</span> - ACM
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m.save("velib_delaunay_legend.html")